# EDS Source Data Lister (SR / FC)

This notebook lists all **Surface Reflectance (SR)** or **Fractional Cover (FC)** source files available under:

- `/home/jovyan/scratch/eds/tiles/{tile}/sr/...`
- `/home/jovyan/scratch/eds/tiles/{tile}/fc/...`

You provide:
- dataset (`sr` or `fc`)
- tile (e.g. `p089r078`)
- date range (`YYYYMMDD` to `YYYYMMDD`)

The notebook prints matching file paths and summary statistics to support quick QA and test bundle selection.


In [5]:
from pathlib import Path

# ===== USER INPUTS =====
dataset = "fc"          # "sr" or "fc"
tile = "p089r078"       # e.g. "p089r078"
start_date = "20230720" # YYYYMMDD
end_date   = "20240831" # YYYYMMDD

# output controls
print_limit = None      # None = print all; or set e.g. 200 to cap console spam
write_to_file = True    # True writes full list to a txt file
out_dir = Path("/home/jovyan/work-easi-eds/exports/lists")
# =======================


In [6]:
import re
from datetime import datetime

def yyyymmdd_ok(s: str) -> bool:
    return bool(re.fullmatch(r"(19|20)\d{6}", s))

def parse_yyyymmdd(s: str) -> datetime:
    return datetime.strptime(s, "%Y%m%d")

def extract_date_from_path(p: Path) -> str | None:
    """
    Extracts first YYYYMMDD from filename.
    """
    m = re.search(r"(19|20)\d{6}", p.name)
    return m.group(0) if m else None

def list_eds_files(dataset: str, tile: str, start_date: str, end_date: str) -> list[Path]:
    if dataset not in {"sr", "fc"}:
        raise ValueError("dataset must be 'sr' or 'fc'")
    if not re.fullmatch(r"p\d{3}r\d{3}", tile.lower()):
        raise ValueError("tile must look like 'p089r078'")
    if not yyyymmdd_ok(start_date) or not yyyymmdd_ok(end_date):
        raise ValueError("start_date and end_date must be YYYYMMDD")

    start_dt = parse_yyyymmdd(start_date)
    end_dt   = parse_yyyymmdd(end_date)

    root = Path(f"/home/jovyan/scratch/eds/tiles/{tile}/{dataset}")
    if not root.exists():
        print(f"[WARN] Root does not exist: {root}")
        return []

    # Find all GeoTIFFs under the dataset root
    candidates = [p for p in root.rglob("*.tif") if p.is_file()]

    # Filter by date in filename
    out = []
    for p in candidates:
        d = extract_date_from_path(p)
        if not d:
            continue
        dt = parse_yyyymmdd(d)
        if start_dt <= dt <= end_dt:
            out.append(p)

    # Sort by date then path for stable output
    out.sort(key=lambda p: (extract_date_from_path(p) or "", str(p)))
    return out


In [7]:
out_dir.mkdir(parents=True, exist_ok=True)

files = list_eds_files(dataset, tile, start_date, end_date)

print(f"[INFO] dataset: {dataset}")
print(f"[INFO] tile: {tile}")
print(f"[INFO] date range requested: {start_date} → {end_date}")
print(f"[INFO] files found: {len(files)}")

if files:
    first = extract_date_from_path(files[0])
    last  = extract_date_from_path(files[-1])
    print(f"[INFO] date range found: {first} → {last}")

print("\nPaths:")
to_print = files if (print_limit is None) else files[:print_limit]
for i, p in enumerate(to_print):
    print(f"[{i}] {p}")

if print_limit is not None and len(files) > print_limit:
    print(f"\n[INFO] Printed first {print_limit} of {len(files)} files (set print_limit=None to print all).")

if write_to_file:
    out_txt = out_dir / f"{dataset}_{tile}_{start_date}_{end_date}.txt"
    out_txt.write_text("\n".join(str(p) for p in files) + "\n", encoding="utf-8")
    print(f"\n[OK] Wrote full list to: {out_txt}")


[INFO] dataset: fc
[INFO] tile: p089r078
[INFO] date range requested: 20230720 → 20240831
[INFO] files found: 66
[INFO] date range found: 20230720 → 20240831

Paths:
[0] /home/jovyan/scratch/eds/tiles/p089r078/fc/2023/202307/galsfc3_p089r078_20230720_fcm6.tif
[1] /home/jovyan/scratch/eds/tiles/p089r078/fc/2023/202307/galsfc3_p089r078_20230720_fcm6_clr.tif
[2] /home/jovyan/scratch/eds/tiles/p089r078/fc/2023/202307/galsfc3_p089r078_20230728_fcm6.tif
[3] /home/jovyan/scratch/eds/tiles/p089r078/fc/2023/202307/galsfc3_p089r078_20230728_fcm6_clr.tif
[4] /home/jovyan/scratch/eds/tiles/p089r078/fc/2023/202308/galsfc3_p089r078_20230805_fcm6.tif
[5] /home/jovyan/scratch/eds/tiles/p089r078/fc/2023/202308/galsfc3_p089r078_20230805_fcm6_clr.tif
[6] /home/jovyan/scratch/eds/tiles/p089r078/fc/2023/202308/galsfc3_p089r078_20230813_fcm6.tif
[7] /home/jovyan/scratch/eds/tiles/p089r078/fc/2023/202308/galsfc3_p089r078_20230813_fcm6_clr.tif
[8] /home/jovyan/scratch/eds/tiles/p089r078/fc/2023/202308/galsfc3

In [10]:
from pathlib import Path
import zipfile

EXPORT_ZIP_DIR = Path("/home/jovyan/work-easi-eds/exports/zips")
EXPORT_ZIP_DIR.mkdir(parents=True, exist_ok=True)

def sidecar_files(p: Path) -> list[Path]:
    """
    Return p plus any existing ancillary files.
    """
    out = [p]

    # Common GDAL sidecars
    candidates = [
        p.with_name(p.name + ".aux.xml"),
        p.with_name(p.name + ".ovr"),
    ]

    # ENVI-style sidecars (rare for FC but harmless)
    candidates.extend([
        p.with_suffix(".hdr"),
        p.with_suffix(".prj"),
        p.with_suffix(".HDR"),
        p.with_suffix(".PRJ"),
    ])

    for c in candidates:
        if c.exists():
            out.append(c)

    return out

zip_name = f"{dataset}_{tile}_{start_date}_{end_date}.zip"
zip_path = EXPORT_ZIP_DIR / zip_name

if zip_path.exists():
    zip_path.unlink()

included = set()

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as z:
    for p in files:
        for f in sidecar_files(Path(p)):
            if f in included:
                continue
            included.add(f)

            # keep directory structure relative to /home/jovyan/scratch/eds
            arcname = f.relative_to("/home/jovyan/scratch/eds")
            z.write(f, arcname=str(arcname))

print(f"[OK] ZIP written: {zip_path}")
print(f"[INFO] Files included (with ancillaries): {len(included)}")
print(f"[INFO] ZIP size (GB): {zip_path.stat().st_size / (1024**3):.3f}")


[OK] ZIP written: /home/jovyan/work-easi-eds/exports/zips/fc_p089r078_20230720_20240831.zip
[INFO] Files included (with ancillaries): 66
[INFO] ZIP size (GB): 4.854
